In [ ]:
# Run this cell if you're using from colab
#!git clone https://github.com/R-Oc-A/HackathonPastryLPV.git
#!pip install https://github.com/R-Oc-A/HackathonPastryLPV/releases/download/wheel/pastrypy-010-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
#import sys
#sys.path.append('/content/HackathonPastryLPV')


In [9]:
import pastrypy as psp
import pulsation_description as plsd
import line_profile_description as lpd
import tomli_w
import os
import polars as pl
import matplotlib.pyplot as plt

# Pulstar configuration
Here you specify the star you'll be modelling as well as the modes of pulsation

In [26]:
#Taken from a Simbad quick Query and from Teltings paper
mode=plsd.Mode(l=2,m=1,
                rel_dr=0.024,
                k=0.03,frequency=5.38,
                phase_offset=0.0,
                rel_dtemp=2.62,
                phase_rel_dtemp=180.0,
                rel_dg=10.0,
                phase_rel_dg=34.0,
                rotation_effects = "PerturbativeCoriolis")
star_data=plsd.StarData(mass=7.4,
                        radius=7.22,
                        effective_temperature=26000.0,
                        v_omega=54.0,
                        inclination_angle=60.0)
time_points=plsd.TimePoints(Uniform=plsd.UniformTime(start=0.0,end=0.0,step=0.01))
mesh=plsd.Mesh(Sphere=plsd.SphericalStar(theta_step=2.0,phi_step=4.0))

pulsconfig=plsd.PulstarConfig(mode_data=[mode],star_data=star_data,time_points=time_points,mesh=mesh)
pulsconfig_dict=pulsconfig.model_dump(exclude_none=True)
puls_toml_string=tomli_w.dumps(pulsconfig_dict)

# Profile configuration
Here you specify the line profile variability you want to observe.

In [27]:
#Taken from a Simbad quick Query
wl_range=lpd.WavelengthRange(start=455.10,end=455.50,step=0.0033)
#path_to_grids="../profile/grids/"
path_to_grids = os.getenv("GRIDS")
grid1=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=25000.0,log_gravity=3.5,filename="t25000g35.txt"),Nadya=None)
grid2=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=25000.0,log_gravity=4.5,filename="t25000g45.txt"),Nadya=None)
grid3=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=27000.0,log_gravity=3.5,filename="t27000g35.txt"),Nadya=None)
grid4=lpd.IntensityGrid(Joris=lpd.JorisGrid(temperature=27000.0,log_gravity=4.5,filename="t27000g45.txt"),Nadya=None)
prof_config=lpd.ProfileConfig(max_velocity=1.0e2,path_to_grids=path_to_grids,wavelength_range=wl_range,intensity_grids=[grid1,grid2,grid3,grid4])

prof_config_dict=prof_config.model_dump(exclude_none=True)

prof_toml_string=tomli_w.dumps(prof_config_dict)


# First run

In [28]:
pulse_df = psp.pulstar(puls_toml_string)

--------------------
--------------------
--------------------
|PULSTARust launched|
--------------------

 +-- Computing surface data for time point number 0 with time stamp 0.000.
----------------------
|PULSTARust Finished |
----------------------


In [30]:
pulse_df.sort("area").tail(5)

coord1,coord2,time,velocity,temperature,log gravity,coschi,area
f64,f64,f64,f64,f64,f64,f64,f64
1.343904,0.069813,0.0,4.298209,26149.800912,3.605476,0.958742,7.423823
1.308997,0.0,0.0,7.4995e-18,26188.967348,3.606912,0.969729,7.435976
1.343904,0.0,0.0,8.1450e-18,26165.675666,3.604882,0.960784,7.439127
1.308997,6.213372,0.0,-4.429915,26206.15324,3.606157,0.967708,7.439585
1.343904,6.213372,0.0,-4.298209,26180.743265,3.604216,0.958742,7.440314


In [31]:
wavelength_df = psp.profile(prof_toml_string,pulse_df)

----------------------------------------
----------------------------------------
[0.0]
min relative dopplershift is 0.999821493584722
max relative dopplershift is 1.000178506415278
creating the spectral grids data structures from csv files
Allocating memory for hypercube in the parameter space
Done
done computing flux
finished collecting fluxes 0
this is the df for mode 0: shape: (5, 5)
┌────────────┬──────────┬──────┬──────────┬───────────┐
│ wavelength ┆ pixel_id ┆ time ┆ flux     ┆ continuum │
│ ---        ┆ ---      ┆ ---  ┆ ---      ┆ ---       │
│ f64        ┆ u32      ┆ f64  ┆ f64      ┆ f64       │
╞════════════╪══════════╪══════╪══════════╪═══════════╡
│ 455.1      ┆ 0        ┆ 0.0  ┆ 2.527687 ┆ 2.528273  │
│ 455.1033   ┆ 1        ┆ 0.0  ┆ 2.52763  ┆ 2.528247  │
│ 455.1066   ┆ 2        ┆ 0.0  ┆ 2.527566 ┆ 2.528222  │
│ 455.1099   ┆ 3        ┆ 0.0  ┆ 2.527498 ┆ 2.528197  │
│ 455.1132   ┆ 4        ┆ 0.0  ┆ 2.527425 ┆ 2.528172  │
└────────────┴──────────┴──────┴──────────┴──────